# Gemma Runtime Smoke-Test

This notebook tests Google Gemma models across three runtimes:
- **Transformers** (Hugging Face, in-notebook Python)
- **Ollama** (external CLI/API)
- **llamafile** (single-file distribution)

## Goals
1. Confirm model runnability across different hardware configurations
2. Collect reliable, comparable timings
3. Provide clear debug signals for failures
4. Generate actionable recommendations for local/sovereign agentic apps

## Models Tested
## Select models to test in Cell 3 "Configuration Section"
- Gemma 3: 1b, 4b, 12b (with Gemma 2 fallbacks)
- All models are instruction-tuned variants (-it)

## Output
- `artifacts/results.jsonl` - Raw test results
- `artifacts/results.csv` - Flattened summary
- `artifacts/report.md` - OpenAI-generated analysis
- `artifacts/report.json` - Machine-readable report

## Setup: Imports and Environment

In [ ]:
# Install core dependencies for Gemma and OpenAI OSS runtime
%pip install -q \
    "jupyter>=1.0.0" \
    "notebook>=7.0.0" \
    "ipykernel>=6.25.0" \
    "transformers>=4.48.0" \
    "torch>=2.4.0" \
    "accelerate>=0.33.0" \
    "huggingface_hub>=0.24.0" \
    "requests>=2.31.0" \
    "psutil>=5.9.0" \
    "pandas>=2.0.0" \
    "openai>=1.40.0" \
    "tqdm>=4.65.0" \
    "bitsandbytes" # Essential for loading 20B models on Colab GPUs

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================================
# CPU AND THREAD GUARDRAILS
# ============================================================================
# Set thread limits BEFORE importing torch/transformers to prevent CPU thrashing

import os
import psutil

# Limit to 4 threads or CPU count, whichever is smaller
CPU_THREADS = min(4, psutil.cpu_count() or 4)
os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_THREADS)

print(f"🔧 Applied CPU thread limits: {CPU_THREADS} threads")
print(f"   OMP_NUM_THREADS={os.environ['OMP_NUM_THREADS']}")
print(f"   MKL_NUM_THREADS={os.environ['MKL_NUM_THREADS']}")
print(f"   OPENBLAS_NUM_THREADS={os.environ['OPENBLAS_NUM_THREADS']}")

# ============================================================================
# STANDARD IMPORTS
# ============================================================================

import sys
import json
import time
import subprocess
import platform
import warnings
import signal
import resource
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from getpass import getpass
import pandas as pd

warnings.filterwarnings('ignore')

# Create output directory
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"\nPython: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"CPU Count: {psutil.cpu_count()} logical, {psutil.cpu_count(logical=False) or 'N/A'} physical")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")


## Configuration Section

All user-configurable parameters are defined here.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Runtime flags - which tests to run
RUN_TRANSFORMERS = True
RUN_OLLAMA = True
RUN_LLAMAFILE = True
RUN_OPENAI_REPORT = True

# ============================================================================
# MODEL CONFIGURATION
# All model identifiers with verification URLs
# ============================================================================

# --- TRANSFORMERS (Hugging Face) ---
# Verify models at: https://huggingface.co/<model_id>
TRANSFORMERS_MODELS = [
    # "google/gemma-3-1b-it",     # https://huggingface.co/google/gemma-3-1b-it
    # "google/gemma-3-4b-it",     # https://huggingface.co/google/gemma-3-4b-it
    # "google/gemma-3-12b-it",  # https://huggingface.co/google/gemma-3-12b-it (large, requires 16GB+ RAM)
    "mistralai/Mistral-7B-Instruct-v0.3",        # https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 (~7 GB bf16)
    # "mistralai/Mistral-Small-24B-Instruct-2501",  # https://huggingface.co/mistralai/Mistral-Small-24B-Instruct-2501 (~12 GB bf16)
]

TRANSFORMERS_FALLBACK_MODELS = [
    # "google/gemma-2-2b-it",   # https://huggingface.co/google/gemma-2-2b-it
    # "google/gemma-2-9b-it",   # https://huggingface.co/google/gemma-2-9b-it
]

# --- OLLAMA ---
# Verify tags at: https://ollama.com/library/gemma3 and https://ollama.com/library/gemma2
# Note: Ollama uses simple tags like "gemma3:1b", not HuggingFace repo IDs
OLLAMA_MODELS = [
    # "gemma3:1b",     # https://ollama.com/library/gemma3:1b (~815MB)
    # "gemma3:4b",     # https://ollama.com/library/gemma3:4b (~3.3GB)
    # "gemma3:12b",  # https://ollama.com/library/gemma3:12b (~8.1GB)
    "hf.co/ggml-org/gpt-oss-20b-GGUF:Q4_K_M",  # ~11 GB, CPU-friendly quantization
]

OLLAMA_FALLBACK_MODELS = [
    # "gemma2:2b",   # https://ollama.com/library/gemma2:2b
    # "gemma2:9b",   # https://ollama.com/library/gemma2:9b
]

# --- LLAMAFILE ---
# Verify repos at: https://huggingface.co/Mozilla/<model>-llamafile
# Files are named like: google_gemma-3-<size>-it-Q6_K.llamafile
LLAMAFILE_MODELS = [
    {
        "repo_id": "Mozilla/gemma-3-1b-it-llamafile",   # https://huggingface.co/Mozilla/gemma-3-1b-it-llamafile
        "filename": "google_gemma-3-1b-it-Q6_K.llamafile"
    },
    {
        "repo_id": "Mozilla/gemma-3-4b-it-llamafile",   # https://huggingface.co/Mozilla/gemma-3-4b-it-llamafile
        "filename": "google_gemma-3-4b-it-Q6_K.llamafile"
    },
    # {
    #     "repo_id": "Mozilla/gemma-3-12b-it-llamafile",  # https://huggingface.co/Mozilla/gemma-3-12b-it-llamafile
    #     "filename": "google_gemma-3-12b-it-Q6_K.llamafile"
    # },
]

LLAMAFILE_FALLBACK_MODELS = [
    # {
    #     "repo_id": "mozilla-ai/gemma-2-2b-it-llamafile",  # https://huggingface.co/mozilla-ai/gemma-2-2b-it-llamafile
    #     "filename": "google_gemma-2-2b-it-Q6_K.llamafile"
    # },
]

# ============================================================================
# TIMEOUTS (in seconds)
# ============================================================================
DOWNLOAD_TIMEOUT_S = 1800  # 30 minutes for large model downloads
FIRST_LOAD_TIMEOUT_S = 600  # 10 minutes for first load into memory
INFERENCE_TIMEOUT_S = 120  # 2 minutes per prompt
PROCESS_SHUTDOWN_TIMEOUT_S = 30  # 30 seconds for cleanup

# ============================================================================
# GENERATION PARAMETERS
# ============================================================================
MAX_TOKENS = 512
TEMPERATURE = 0.7
REPETITION_PENALTY = 1.2  # Prevent repetitive output (1.0 = no penalty, higher = more penalty)
SEED = 42

# ============================================================================
# PROMPTS
# ============================================================================
SHORT_PROMPT = "What is the capital of France?"

COMPLEX_PROMPT = """Extract information from the following text and output as JSON with keys 'name', 'age', 'city', and 'occupation':

Text: John Smith is a 35-year-old software engineer living in San Francisco.

Output the result as valid JSON only, no additional text."""

PROMPTS = [
    {"name": "short", "text": SHORT_PROMPT},
    {"name": "complex", "text": COMPLEX_PROMPT},
]

# ============================================================================
# OPENAI CONFIGURATION
# ============================================================================
OPENAI_MODEL = "gpt-4o"  # Use gpt-4o or latest available

# ============================================================================
# OUTPUT PATHS
# ============================================================================
RESULTS_JSONL = ARTIFACTS_DIR / "results.jsonl"
RESULTS_CSV = ARTIFACTS_DIR / "results.csv"
REPORT_MD = ARTIFACTS_DIR / "report.md"
REPORT_JSON = ARTIFACTS_DIR / "report.json"


print("✓ Configuration loaded")

## Token Management

Prompt for required API tokens if not set in environment.

In [ ]:
# ============================================================================
# TOKEN MANAGEMENT — load from .env first, then fall back to interactive prompt
# ============================================================================
# ⚠️  WARNING: If you enter keys interactively below, they will persist in
#     this notebook's output cells. Run the CLEANUP cell at the very end
#     before committing to git, or your keys WILL be exposed!
#
# Recommended: store keys in .env (see .env.example) instead of typing them.
# ============================================================================

# Try loading from .env file
try:
    from dotenv import load_dotenv
    # Look in workspace root (one level up from notebooks/)
    _dotenv_path = Path('../.env')
    if _dotenv_path.exists():
        load_dotenv(_dotenv_path)
        print(f"✓ Loaded .env from {_dotenv_path.resolve()}")
    else:
        print(f"ℹ No .env file found at {_dotenv_path.resolve()}")
        print(f"  Tip: cp .env.example .env  and fill in your keys")
except ImportError:
    print("ℹ python-dotenv not installed — skipping .env loading")
    print("  Install with: pip install python-dotenv")

# Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and RUN_TRANSFORMERS:
    print("\n⚠️  Hugging Face token not found in environment or .env")
    HF_TOKEN = getpass("Enter HF_TOKEN (or press Enter to skip gated models): ")
    if not HF_TOKEN:
        print("⚠ No HF token provided. Transformers tests will be skipped if gated models are encountered.")
        RUN_TRANSFORMERS = False
    else:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print("✓ HF token set (from interactive input)")
        print("  ⚠️  Remember to run the CLEANUP cell before committing!")
else:
    if HF_TOKEN:
        print(f"✓ HF token loaded (from environment/dotenv, ends with ...{HF_TOKEN[-4:]})")

# OpenAI API key
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
if not OPENAI_API_KEY and RUN_OPENAI_REPORT:
    print("\n⚠️  OpenAI API key not found in environment or .env")
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (or press Enter to skip report generation): ")
    if not OPENAI_API_KEY:
        print("⚠ No OpenAI API key provided. Report generation will be skipped.")
        RUN_OPENAI_REPORT = False
    else:
        os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
        print("✓ OpenAI API key set (from interactive input)")
        print("  ⚠️  Remember to run the CLEANUP cell before committing!")
else:
    if OPENAI_API_KEY:
        print(f"✓ OpenAI API key loaded (from environment/dotenv, ends with ...{OPENAI_API_KEY[-4:]})")

## Utility Functions

In [ ]:
# ============================================================================
# STREAMING SUBPROCESS HELPER
# ============================================================================

def run_subprocess_with_streaming(
    cmd: List[str],
    timeout_s: int,
    heartbeat_interval_s: int = 30,
    description: str = ""
) -> Tuple[str, str, int, float]:
    """
    Run a subprocess with streaming output and heartbeat logging.
    
    Args:
        cmd: Command and arguments to run
        timeout_s: Maximum time to wait for process
        heartbeat_interval_s: Print heartbeat if no output for this many seconds
        description: Human-readable description for logging
    
    Returns:
        (stdout, stderr, returncode, elapsed_time)
    """
    import select
    
    start_time = time.time()
    last_output_time = start_time
    stdout_lines = []
    stderr_lines = []
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {description or ' '.join(cmd)}")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,  # Line buffered
        )
        
        # Non-blocking read loop
        while True:
            current_time = time.time()
            elapsed = current_time - start_time
            
            # Check timeout
            if elapsed > timeout_s:
                print(f"\n[{datetime.now().strftime('%H:%M:%S')}] ⏱️ TIMEOUT after {elapsed:.1f}s - terminating process")
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠️ Process did not terminate cleanly")
                break
            
            # Check if process is still running
            if process.poll() is not None:
                # Process finished - read remaining output
                for line in process.stdout:
                    line = line.rstrip()
                    if line:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}")
                        stdout_lines.append(line)
                for line in process.stderr:
                    line = line.rstrip()
                    if line:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] STDERR: {line}")
                        stderr_lines.append(line)
                break
            
            # Try to read output (non-blocking on Unix)
            try:
                # Use select on Unix systems for non-blocking I/O
                if hasattr(select, 'select'):
                    readable, _, _ = select.select([process.stdout, process.stderr], [], [], 0.5)
                    
                    if process.stdout in readable:
                        line = process.stdout.readline()
                        if line:
                            line = line.rstrip()
                            print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}")
                            stdout_lines.append(line)
                            last_output_time = current_time
                    
                    if process.stderr in readable:
                        line = process.stderr.readline()
                        if line:
                            line = line.rstrip()
                            print(f"[{datetime.now().strftime('%H:%M:%S')}] STDERR: {line}")
                            stderr_lines.append(line)
                            last_output_time = current_time
                else:
                    # Fallback for Windows - just wait
                    time.sleep(0.5)
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Error reading output: {e}")
                time.sleep(0.5)
            
            # Print heartbeat if no output for a while
            if current_time - last_output_time >= heartbeat_interval_s:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] 💓 [heartbeat] still waiting for {description} — {elapsed:.0f}s elapsed, no new output for {current_time - last_output_time:.0f}s")
                last_output_time = current_time  # Reset to avoid spam
        
        elapsed_time = time.time() - start_time
        returncode = process.returncode if process.returncode is not None else -1
        
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ Process completed in {elapsed_time:.1f}s (exit code: {returncode})")
        
        return ('\n'.join(stdout_lines), '\n'.join(stderr_lines), returncode, elapsed_time)
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Exception: {e}")
        try:
            process.terminate()
            process.wait(timeout=10)
        except:
            pass
        return ('', str(e), -1, elapsed_time)


# ============================================================================
# MEMORY GUARDRAILS
# ============================================================================

# Model size estimates in GB (fp32 / fp16)
MODEL_SIZE_ESTIMATES = {
    '1b': {'fp32': 2.5, 'fp16': 1.3},
    '2b': {'fp32': 5.0, 'fp16': 2.5},
    '4b': {'fp32': 9.0, 'fp16': 4.5},
    '7b': {'fp32': 14.0, 'fp16': 7.0},
    '9b': {'fp32': 20.0, 'fp16': 10.0},
    '12b': {'fp32': 27.0, 'fp16': 13.5},
    '24b': {'fp32': 48.0, 'fp16': 24.0},
}

def estimate_model_size(model_id: str) -> float:
    """Estimate model memory footprint in GB (assumes fp16)."""
    model_lower = model_id.lower()
    
    # Check sizes from largest to smallest to avoid substring matches
    for size_key in ['12b', '9b', '4b', '2b', '1b']:
        if size_key in model_lower:
            return MODEL_SIZE_ESTIMATES[size_key]['fp16']
    
    # Default to 2GB for unknown models
    return 2.0

def check_resources(model_id: str) -> Tuple[bool, str]:
    """
    Check if system has enough resources to load a model.
    
    Returns:
        (is_ok, message)
    """
    vm = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    available_gb = vm.available / (1024**3)
    total_gb = vm.total / (1024**3)
    swap_pct = swap.percent
    
    estimated_size_gb = estimate_model_size(model_id)
    
    # Check if model would use > 80% of available RAM
    threshold_gb = available_gb * 0.8
    
    print(f"\n{'='*60}")
    print(f"Resource Check: {model_id}")
    print(f"{'='*60}")
    print(f"Estimated model size: {estimated_size_gb:.1f} GB (fp16)")
    print(f"Available RAM: {available_gb:.1f} GB / {total_gb:.1f} GB total")
    print(f"Swap usage: {swap_pct:.1f}%")
    print(f"Threshold (80% of available): {threshold_gb:.1f} GB")
    
    # Check swap first
    if swap_pct > 50:
        msg = f"⚠️ WARNING: System is already swapping ({swap_pct:.1f}% swap used)"
        print(msg)
        print(f"{'='*60}\n")
        return False, msg
    
    # Check if model would fit
    if estimated_size_gb > threshold_gb:
        msg = f"🛑 BLOCKED: Model size ({estimated_size_gb:.1f} GB) exceeds 80% of available RAM ({threshold_gb:.1f} GB)"
        print(msg)
        print(f"{'='*60}\n")
        return False, msg
    
    # All good
    msg = f"✅ Resources OK: {estimated_size_gb:.1f} GB needed, {available_gb:.1f} GB available"
    print(msg)
    print(f"{'='*60}\n")
    return True, msg


# ============================================================================
# EXISTING UTILITY FUNCTIONS
# ============================================================================

def get_environment_fingerprint() -> Dict[str, Any]:
    """Collect environment metadata for reproducibility."""
    fingerprint = {
        "timestamp": datetime.now().isoformat(),
        "os": platform.platform(),
        "python_version": sys.version,
        "cpu_count": psutil.cpu_count(),
        "ram_gb": psutil.virtual_memory().total / (1024**3),
    }
    
    # Try to get package versions
    try:
        import torch
        fingerprint["torch_version"] = torch.__version__
        fingerprint["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            fingerprint["cuda_version"] = torch.version.cuda
            fingerprint["gpu_name"] = torch.cuda.get_device_name(0)
        if hasattr(torch.backends, 'mps'):
            fingerprint["mps_available"] = torch.backends.mps.is_available()
    except ImportError:
        fingerprint["torch_version"] = "not_installed"
    
    try:
        import transformers
        fingerprint["transformers_version"] = transformers.__version__
    except ImportError:
        fingerprint["transformers_version"] = "not_installed"
    
    return fingerprint


def log_phase(phase: str, **kwargs):
    """Log a timestamped phase marker."""
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "phase": phase,
        **kwargs
    }
    print(json.dumps(log_entry))
    return log_entry


def get_memory_info() -> Dict[str, float]:
    """Get current memory usage."""
    vm = psutil.virtual_memory()
    return {
        "total_gb": vm.total / (1024**3),
        "available_gb": vm.available / (1024**3),
        "used_gb": vm.used / (1024**3),
        "percent": vm.percent,
    }


def clear_memory():
    """Aggressively clear memory."""
    log_phase("CLEAR_START", memory=get_memory_info())
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except ImportError:
        pass
    
    import gc
    gc.collect()
    
    log_phase("CLEAR_DONE", memory=get_memory_info())


def save_result(result: Dict[str, Any]):
    """Append result to JSONL file."""
    with open(RESULTS_JSONL, 'a') as f:
        f.write(json.dumps(result) + '\n')



def count_tokens_simple(text: str) -> int:
    """
    Simple token counter that approximates tokens without loading a tokenizer.
    Uses a heuristic: ~4 characters per token (typical for English text).
    This is used for throughput metrics when a full tokenizer is not available.
    """
    if not text:
        return 0
    # Approximate: 1 token ≈ 4 characters (conservative estimate)
    return max(1, len(text) // 4)


def create_result_record(
    runtime: str,
    model: str,
    status: str,
    prompt_name: str = "",
    prompt_text: str = "",
    output: str = "",
    duration_s: float = 0.0,
    device: str = "",
    error: str = "",
    **kwargs
) -> Dict[str, Any]:
    """Create a standardized result record."""
    return {
        "timestamp": datetime.now().isoformat(),
        "runtime": runtime,
        "model": model,
        "status": status,  # SUCCESS, FAILED, SKIPPED
        "prompt_name": prompt_name,
        "prompt_text": prompt_text,
        "output": output,
        "duration_s": duration_s,
        "device": device,
        "error": error,
        **kwargs
    }

print("✓ Utility functions loaded")

## Environment Fingerprint

In [ ]:
env_fingerprint = get_environment_fingerprint()
print(json.dumps(env_fingerprint, indent=2))

## System Diagnostics

Detailed system resource analysis to understand performance constraints.


In [ ]:
# ============================================================================
# SYSTEM DIAGNOSTICS
# ============================================================================

print("\n" + "="*80)
print("SYSTEM DIAGNOSTICS")
print("="*80 + "\n")

diagnostics = {}

# Physical vs logical cores
try:
    diagnostics['cpu_logical_cores'] = psutil.cpu_count(logical=True)
    diagnostics['cpu_physical_cores'] = psutil.cpu_count(logical=False) or 'N/A'
except Exception as e:
    diagnostics['cpu_cores'] = f'Error: {e}'

# CPU frequency
try:
    freq = psutil.cpu_freq()
    if freq:
        diagnostics['cpu_freq_current_mhz'] = f"{freq.current:.0f}"
        diagnostics['cpu_freq_min_mhz'] = f"{freq.min:.0f}" if freq.min else 'N/A'
        diagnostics['cpu_freq_max_mhz'] = f"{freq.max:.0f}" if freq.max else 'N/A'
    else:
        diagnostics['cpu_freq'] = 'N/A'
except Exception as e:
    diagnostics['cpu_freq'] = f'Error: {e}'

# RAM
try:
    vm = psutil.virtual_memory()
    diagnostics['ram_total_gb'] = f"{vm.total / (1024**3):.1f}"
    diagnostics['ram_available_gb'] = f"{vm.available / (1024**3):.1f}"
    diagnostics['ram_used_pct'] = f"{vm.percent:.1f}%"
except Exception as e:
    diagnostics['ram'] = f'Error: {e}'

# Swap
try:
    swap = psutil.swap_memory()
    diagnostics['swap_total_gb'] = f"{swap.total / (1024**3):.1f}"
    diagnostics['swap_used_gb'] = f"{swap.used / (1024**3):.1f}"
    diagnostics['swap_free_gb'] = f"{swap.free / (1024**3):.1f}"
    diagnostics['swap_used_pct'] = f"{swap.percent:.1f}%"
except Exception as e:
    diagnostics['swap'] = f'Error: {e}'

# Disk I/O throughput estimate (100MB sequential read)
try:
    import tempfile
    test_size = 100 * 1024 * 1024  # 100MB
    test_file = tempfile.NamedTemporaryFile(delete=False)
    
    # Write test
    write_start = time.time()
    test_file.write(os.urandom(test_size))
    test_file.flush()
    os.fsync(test_file.fileno())
    write_time = time.time() - write_start
    
    test_file.close()
    
    # Read test
    read_start = time.time()
    with open(test_file.name, 'rb') as f:
        _ = f.read()
    read_time = time.time() - read_start
    
    os.unlink(test_file.name)
    
    write_throughput = test_size / (1024**2) / write_time  # MB/s
    read_throughput = test_size / (1024**2) / read_time  # MB/s
    
    diagnostics['disk_write_mb_s'] = f"{write_throughput:.0f}"
    diagnostics['disk_read_mb_s'] = f"{read_throughput:.0f}"
except Exception as e:
    diagnostics['disk_io'] = f'Error: {e}'

# Memory bandwidth estimate (512MB memcpy)
try:
    test_size = 512 * 1024 * 1024  # 512MB
    data = bytearray(test_size)
    
    copy_start = time.time()
    copy_data = bytearray(data)
    copy_time = time.time() - copy_start
    
    bandwidth_gb_s = test_size / (1024**3) / copy_time
    diagnostics['memory_bandwidth_gb_s'] = f"{bandwidth_gb_s:.2f}"
    
    # Cleanup
    del data, copy_data
except Exception as e:
    diagnostics['memory_bandwidth'] = f'Error: {e}'

# GPU memory (if CUDA)
try:
    import torch
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        diagnostics['gpu_memory_free_gb'] = f"{free / (1024**3):.1f}"
        diagnostics['gpu_memory_total_gb'] = f"{total / (1024**3):.1f}"
    else:
        diagnostics['gpu_memory'] = 'N/A (no CUDA)'
except Exception as e:
    diagnostics['gpu_memory'] = 'N/A'

# CPU governor / scaling
try:
    governor_file = Path('/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor')
    if governor_file.exists():
        diagnostics['cpu_governor'] = governor_file.read_text().strip()
    else:
        diagnostics['cpu_governor'] = 'N/A'
except Exception as e:
    diagnostics['cpu_governor'] = 'N/A'

# Container memory limit
try:
    # Try cgroup v2
    mem_max = Path('/sys/fs/cgroup/memory.max')
    if mem_max.exists():
        limit = mem_max.read_text().strip()
        if limit != 'max':
            diagnostics['container_mem_limit_gb'] = f"{int(limit) / (1024**3):.1f}"
        else:
            diagnostics['container_mem_limit'] = 'unlimited'
    else:
        # Try cgroup v1
        mem_limit = Path('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        if mem_limit.exists():
            limit = int(mem_limit.read_text().strip())
            if limit < 9223372036854771712:  # Not unlimited
                diagnostics['container_mem_limit_gb'] = f"{limit / (1024**3):.1f}"
            else:
                diagnostics['container_mem_limit'] = 'unlimited'
        else:
            diagnostics['container_mem_limit'] = 'N/A'
except Exception as e:
    diagnostics['container_mem_limit'] = 'N/A'

# Container CPU limit
try:
    # Try cgroup v2
    cpu_max = Path('/sys/fs/cgroup/cpu.max')
    if cpu_max.exists():
        content = cpu_max.read_text().strip()
        if content != 'max':
            parts = content.split()
            if len(parts) == 2:
                quota, period = parts
                if quota != 'max':
                    diagnostics['container_cpu_limit'] = f"{int(quota)/int(period):.2f} cores"
                else:
                    diagnostics['container_cpu_limit'] = 'unlimited'
            else:
                diagnostics['container_cpu_limit'] = 'unlimited'
        else:
            diagnostics['container_cpu_limit'] = 'unlimited'
    else:
        # Try cgroup v1
        cpu_quota = Path('/sys/fs/cgroup/cpu/cpu.cfs_quota_us')
        cpu_period = Path('/sys/fs/cgroup/cpu/cpu.cfs_period_us')
        if cpu_quota.exists() and cpu_period.exists():
            quota = int(cpu_quota.read_text().strip())
            period = int(cpu_period.read_text().strip())
            if quota > 0:
                diagnostics['container_cpu_limit'] = f"{quota/period:.2f} cores"
            else:
                diagnostics['container_cpu_limit'] = 'unlimited'
        else:
            diagnostics['container_cpu_limit'] = 'N/A'
except Exception as e:
    diagnostics['container_cpu_limit'] = 'N/A'

# Thermal throttling
try:
    thermal_zones = list(Path('/sys/class/thermal').glob('thermal_zone*/temp'))
    if thermal_zones:
        temps = []
        for zone in thermal_zones[:4]:  # Only check first 4
            try:
                temp = int(zone.read_text().strip()) / 1000.0
                temps.append(f"{temp:.0f}°C")
            except:
                pass
        diagnostics['thermal_zones'] = ', '.join(temps) if temps else 'N/A'
    else:
        diagnostics['thermal_zones'] = 'N/A'
except Exception as e:
    diagnostics['thermal_zones'] = 'N/A'

# ulimit -v (virtual memory limit)
try:
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    if soft == resource.RLIM_INFINITY:
        diagnostics['ulimit_virtual_memory'] = 'unlimited'
    else:
        diagnostics['ulimit_virtual_memory_gb'] = f"{soft / (1024**3):.1f}"
except Exception as e:
    diagnostics['ulimit_virtual_memory'] = 'N/A'

# OOM scorer score
try:
    oom_score_file = Path(f'/proc/{os.getpid()}/oom_score')
    if oom_score_file.exists():
        diagnostics['oom_score'] = oom_score_file.read_text().strip()
    else:
        diagnostics['oom_score'] = 'N/A'
except Exception as e:
    diagnostics['oom_score'] = 'N/A'

# Print formatted table
print(f"{'Diagnostic':<35} {'Value':<30}")
print("-" * 65)
for key, value in diagnostics.items():
    # Format key (convert snake_case to Title Case)
    display_key = key.replace('_', ' ').title()
    print(f"{display_key:<35} {value:<30}")

print("\n" + "="*80 + "\n")


## Transformers Runtime Tests

Test models using Hugging Face Transformers library.

**Success criteria:**
- Model loads successfully
- Inference completes within timeout
- Output is generated for each prompt
- Device info is logged (CPU, CUDA, MPS)

In [ ]:
def test_transformers_model(model_id: str) -> List[Dict[str, Any]]:
    """Test a single Transformers model with all prompts."""
    results = []
    model = None
    tokenizer = None
    
    # Check if model can fit in memory
    is_ok, resource_msg = check_resources(model_id)
    if not is_ok:
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error="Insufficient resources: model estimated to exceed available memory",
        ))
        return results
    
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        
        # Determine device
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
        
        log_phase("LOAD_START", model=model_id, device=device, memory=get_memory_info())
        
        # Load model
        print(f"Loading {model_id}...")
        start_time = time.time()
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
            # Use bfloat16 on all devices to halve memory vs fp32 (works on modern PyTorch CPU)
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                torch_dtype=torch.bfloat16,
                device_map="auto" if device == "cuda" else None,
                low_cpu_mem_usage=True,  # Load weights incrementally to reduce peak RAM
            )
            if device != "cuda":
                model = model.to(device)
            model.eval()
            load_duration = time.time() - start_time
            log_phase("LOAD_DONE", model=model_id, duration_s=load_duration, memory=get_memory_info())
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("LOAD_FAILED", model=model_id, error=error_msg)
            results.append(create_result_record(
                runtime="transformers",
                model=model_id,
                status="FAILED",
                error=error_msg,
                device=device,
            ))
            return results
        
        # Run inference for each prompt
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]
            
            try:
                print(f"Running inference on {model_id} with prompt '{prompt_name}'...")
                log_phase("INFER_START", model=model_id, prompt=prompt_name)
                
                # Capture memory before inference
                mem_before = get_memory_info()
                mem_before_mb = mem_before['used_gb'] * 1024
                
                start_time = time.time()
                inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=MAX_TOKENS,
                        temperature=TEMPERATURE,
                        do_sample=True,
                        repetition_penalty=REPETITION_PENALTY,
                    )
                
                output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                duration = time.time() - start_time
                
                # Capture memory after inference
                mem_after = get_memory_info()
                mem_after_mb = mem_after['used_gb'] * 1024
                mem_delta_mb = mem_after_mb - mem_before_mb
                
                log_phase("INFER_DONE", model=model_id, prompt=prompt_name, duration_s=duration)
                
                # Calculate throughput metrics
                prompt_tokens = len(inputs['input_ids'][0])
                output_tokens = len(outputs[0])
                tokens_generated = output_tokens - prompt_tokens
                tokens_per_sec = tokens_generated / duration if duration > 0 else 0.0
                
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=output_text,
                    duration_s=duration,
                    device=device,
                    mem_before_mb=mem_before_mb,
                    mem_after_mb=mem_after_mb,
                    mem_delta_mb=mem_delta_mb,
                    prompt_tokens=prompt_tokens,
                    tokens_generated=tokens_generated,
                    tokens_per_sec=tokens_per_sec,
                ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                    device=device,
                ))
    
    except ImportError as e:
        error_msg = f"Import error: {str(e)}"
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error=error_msg,
        ))
    finally:
        # Cleanup
        del model
        del tokenizer
        clear_memory()
    
    return results


if RUN_TRANSFORMERS:
    print("\n" + "="*80)
    print("TRANSFORMERS RUNTIME TESTS")
    print("="*80 + "\n")
    
    # Try primary models first
    all_models = TRANSFORMERS_MODELS + TRANSFORMERS_FALLBACK_MODELS
    
    for model_id in all_models:
        print(f"\nTesting: {model_id}")
        results = test_transformers_model(model_id)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ Transformers tests skipped")


## Ollama Runtime Tests

Test models using Ollama CLI.

**Success criteria:**
- Ollama is installed and server is reachable
- Model pulls successfully (or is already available)
- Inference completes within timeout
- Output is captured correctly

## Ollama Installation Check

Ollama must be installed and running before these tests can execute.


## Ollama Setup

Install Ollama and start the background server so the tests below can run.

In [ ]:
import subprocess, shutil, time

# ── Step 1: Install Ollama if not already present ──
if not shutil.which("ollama"):
    print("📦 Installing Ollama …")
    result = subprocess.run(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Ollama install failed:\n{result.stderr}")
    print("✅ Ollama installed")
else:
    print("✅ Ollama already installed")

# ── Step 2: Start the Ollama server in the background ──
# Check if the server is already running
try:
    subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5, check=True)
    print("✅ Ollama server already running")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("🚀 Starting Ollama server in the background …")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    # ── Step 3: Wait until the server is responsive ──
    for i in range(30):
        try:
            subprocess.run(
                ["ollama", "list"],
                capture_output=True, text=True, timeout=5, check=True,
            )
            print(f"✅ Ollama server ready (took ~{i+1}s)")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama server did not become ready within 30 s")

In [ ]:
# Define check_ollama_available here so the cell is self-contained
def check_ollama_available() -> tuple:
    """Check if Ollama is installed and server is running."""
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            return True, "Ollama is available"
        else:
            return False, f"Ollama CLI error: {result.stderr}"
    except FileNotFoundError:
        return False, "Ollama CLI not found"
    except subprocess.TimeoutExpired:
        return False, "Ollama CLI timeout"
    except Exception as e:
        return False, f"Ollama check error: {str(e)}"

# Check if Ollama is installed
ollama_available, ollama_msg = check_ollama_available()

if not ollama_available:
    print("⚠️ OLLAMA NOT AVAILABLE")
    print(f"   Reason: {ollama_msg}")
    print("\n📋 Installation Instructions:")
    print("\n   Option 1: Install Ollama manually")
    print("   ---------------------------------")
    print("   Visit: https://ollama.com/download")
    print("   Or run: curl -fsSL https://ollama.com/install.sh | sh")
    print("   Then start the server: ollama serve")
    print("\n   Option 2: Auto-install (Linux/macOS only)")
    print("   -------------------------------------------")
    print("   Uncomment and run the following line:")
    print("   # !curl -fsSL https://ollama.com/install.sh | sh")
    print("\n   After installation, start the Ollama server in a separate terminal:")
    print("   $ ollama serve")
    print("\n   Then re-run this cell to verify installation.")
    print("\n   Ollama tests will be SKIPPED until Ollama is available.")
else:
    print("✅ Ollama is installed and available")
    print(f"   {ollama_msg}")

In [ ]:
import requests as _requests

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_COLD_START_TIMEOUT_S = 300  # 5 min for first model load


def check_ollama_available() -> Tuple[bool, str]:
    """Check if Ollama server is reachable via REST API."""
    try:
        r = _requests.get(OLLAMA_BASE_URL, timeout=5)
        if r.status_code == 200:
            return True, "Ollama is available"
        return False, f"Ollama HTTP {r.status_code}"
    except Exception as e:
        return False, f"Ollama not reachable: {e}"


def _ollama_pull(model_tag: str, timeout_s: int) -> Tuple[bool, str, float]:
    """Pull a model via REST API. Returns (ok, message, elapsed)."""
    t0 = time.time()
    try:
        r = _requests.post(
            f"{OLLAMA_BASE_URL}/api/pull",
            json={"name": model_tag},
            timeout=timeout_s,
            stream=True,
        )
        last_status = ""
        for line in r.iter_lines():
            if line:
                data = json.loads(line)
                status = data.get("status", "")
                if status != last_status:
                    print(f"  pull: {status}")
                    last_status = status
                if data.get("error"):
                    return False, data["error"], time.time() - t0
        return True, "ok", time.time() - t0
    except Exception as e:
        return False, str(e), time.time() - t0


def _ollama_generate(
    model_tag: str,
    prompt: str,
    timeout_s: int,
    temperature: float = TEMPERATURE,
    max_tokens: int = MAX_TOKENS,
) -> Dict[str, Any]:
    """
    Call Ollama REST API for generation (non-streaming).
    Returns the full JSON response dict, or a dict with 'error' key on failure.
    """
    try:
        r = _requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": model_tag,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": temperature,
                    "num_predict": max_tokens,
                    "repeat_penalty": REPETITION_PENALTY,
                    "seed": SEED,
                },
            },
            timeout=timeout_s,
        )
        if r.status_code != 200:
            return {"error": f"HTTP {r.status_code}: {r.text[:200]}"}
        return r.json()
    except _requests.exceptions.Timeout:
        return {"error": f"Timeout after {timeout_s}s"}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {str(e)[:200]}"}


def test_ollama_model(model_tag: str) -> List[Dict[str, Any]]:
    """Test a single Ollama model with all prompts via REST API."""
    results = []

    # Resource check
    is_ok, resource_msg = check_resources(model_tag)
    if not is_ok:
        results.append(create_result_record(
            runtime="ollama", model=model_tag, status="SKIPPED", error=resource_msg,
        ))
        return results

    try:
        # ── Pull model ──
        print(f"Pulling {model_tag}...")
        log_phase("DOWNLOAD_START", model=model_tag, runtime="ollama")
        pull_ok, pull_msg, pull_elapsed = _ollama_pull(model_tag, DOWNLOAD_TIMEOUT_S)
        if not pull_ok:
            error_msg = f"Pull failed: {pull_msg}"
            log_phase("DOWNLOAD_FAILED", model=model_tag, error=error_msg)
            results.append(create_result_record(
                runtime="ollama", model=model_tag, status="FAILED", error=error_msg,
            ))
            return results
        print(f"✅ Pull done in {pull_elapsed:.1f}s")
        log_phase("DOWNLOAD_DONE", model=model_tag)

        # ── Warm-up: load model into memory (cold-start) ──
        print(f"🔥 Warming up {model_tag} (timeout: {OLLAMA_COLD_START_TIMEOUT_S}s)...")
        warmup = _ollama_generate(model_tag, "hi", OLLAMA_COLD_START_TIMEOUT_S)
        if "error" in warmup:
            print(f"⚠️ Warm-up issue: {warmup['error']} — continuing anyway")
        else:
            warmup_dur = warmup.get("total_duration", 0) / 1e9  # nanoseconds → seconds
            print(f"✅ Model loaded in {warmup_dur:.1f}s — starting timed inference")

        # ── Inference for each prompt ──
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]

            try:
                print(f"Running inference on {model_tag} with prompt '{prompt_name}'...")
                log_phase("INFER_START", model=model_tag, prompt=prompt_name)

                mem_before = get_memory_info()
                mem_before_mb = mem_before['used_gb'] * 1024

                start_time = time.time()
                resp = _ollama_generate(model_tag, prompt_text, INFERENCE_TIMEOUT_S)
                duration = time.time() - start_time

                mem_after = get_memory_info()
                mem_after_mb = mem_after['used_gb'] * 1024
                mem_delta_mb = mem_after_mb - mem_before_mb

                if "error" in resp:
                    error_msg = f"Inference failed: {resp['error']}"
                    log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                    results.append(create_result_record(
                        runtime="ollama", model=model_tag, status="FAILED",
                        prompt_name=prompt_name, prompt_text=prompt_text, error=error_msg,
                    ))
                else:
                    output_text = resp.get("response", "")
                    # Ollama reports durations in nanoseconds
                    total_ns = resp.get("total_duration", 0)
                    eval_count = resp.get("eval_count", 0)
                    eval_ns = resp.get("eval_duration", 0)
                    prompt_eval_count = resp.get("prompt_eval_count", 0)
                    tokens_per_sec = (eval_count / (eval_ns / 1e9)) if eval_ns > 0 else 0.0

                    log_phase("INFER_DONE", model=model_tag, prompt=prompt_name, duration_s=duration)
                    print(f"  ✅ {eval_count} tokens in {duration:.1f}s ({tokens_per_sec:.1f} tok/s)")

                    results.append(create_result_record(
                        runtime="ollama", model=model_tag, status="SUCCESS",
                        prompt_name=prompt_name, prompt_text=prompt_text,
                        output=output_text, duration_s=duration,
                        mem_before_mb=mem_before_mb, mem_after_mb=mem_after_mb,
                        mem_delta_mb=mem_delta_mb,
                        prompt_tokens=prompt_eval_count,
                        tokens_generated=eval_count,
                        tokens_per_sec=tokens_per_sec,
                        ollama_total_duration_ns=total_ns,
                        ollama_eval_duration_ns=eval_ns,
                    ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="ollama", model=model_tag, status="FAILED",
                    prompt_name=prompt_name, prompt_text=prompt_text, error=error_msg,
                ))
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)[:200]}"
        results.append(create_result_record(
            runtime="ollama", model=model_tag, status="FAILED", error=error_msg,
        ))
    finally:
        clear_memory()

    return results


# ── Run Ollama tests ──
if RUN_OLLAMA:
    print("\n" + "="*80)
    print("OLLAMA RUNTIME TESTS")
    print("="*80 + "\n")

    available, message = check_ollama_available()
    print(f"Ollama status: {message}\n")

    if available:
        all_models = OLLAMA_MODELS + OLLAMA_FALLBACK_MODELS
        for model_tag in all_models:
            print(f"\nTesting: {model_tag}")
            results = test_ollama_model(model_tag)
            for result in results:
                save_result(result)
                print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
            print()
    else:
        print(f"⚠ Ollama tests skipped: {message}")
        for model_tag in OLLAMA_MODELS:
            save_result(create_result_record(
                runtime="ollama", model=model_tag, status="SKIPPED", error=message,
            ))

    # ── Shut down Ollama server to free memory for remaining tests ──
    print("\n🛑 Shutting down Ollama server...")
    try:
        import signal as _sig
        # Find and kill ollama serve processes
        for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
            try:
                cmdline = proc.info.get('cmdline') or []
                if 'ollama' in ' '.join(cmdline) and 'serve' in ' '.join(cmdline):
                    proc.terminate()
                    proc.wait(timeout=10)
                    print(f"  ✅ Stopped ollama serve (PID {proc.info['pid']})")
            except (psutil.NoSuchProcess, psutil.TimeoutExpired):
                try:
                    proc.kill()
                    print(f"  ⚠️ Force-killed ollama serve (PID {proc.info['pid']})")
                except psutil.NoSuchProcess:
                    pass
            except Exception:
                pass
        clear_memory()
        print("✅ Ollama server stopped, memory freed")
    except Exception as e:
        print(f"⚠️ Could not stop Ollama server: {e}")
else:
    print("⚠ Ollama tests skipped (disabled in config)")

## llamafile Runtime Tests

Test models using llamafile executables.

**Success criteria:**
- llamafile downloads successfully
- Executable permissions are set correctly
- Model runs and produces output
- Process terminates cleanly

In [ ]:
def download_llamafile(config: dict) -> Tuple[Optional[Path], str]:
    """Download a llamafile from Hugging Face.
    
    Args:
        config: Dict with 'repo_id' and 'filename' keys
    
    Returns:
        (path, error_message) tuple
    """
    try:
        from huggingface_hub import hf_hub_download
        
        repo_id = config['repo_id']
        filename = config['filename']
        
        # Look for .llamafile in the repo
        log_phase("DOWNLOAD_START", model=repo_id, runtime="llamafile")
        
        print(f"Downloading {repo_id}/{filename}... (timeout: {DOWNLOAD_TIMEOUT_S}s)")
        
        try:
            # Download the llamafile
            llamafile_path = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                token=HF_TOKEN,
                cache_dir=str(ARTIFACTS_DIR / "llamafile_cache"),
            )
            
            # Make executable
            import stat
            path = Path(llamafile_path)
            path.chmod(path.stat().st_mode | stat.S_IEXEC)
            
            log_phase("DOWNLOAD_DONE", model=repo_id)
            return path, ""
        except Exception as e:
            error_msg = f"Download failed: {str(e)[:200]}"
            log_phase("DOWNLOAD_FAILED", model=repo_id, error=error_msg)
            return None, error_msg
    except ImportError:
        return None, "huggingface_hub not installed"


def test_llamafile_model(config: dict) -> List[Dict[str, Any]]:
    """Test a single llamafile model with all prompts."""
    results = []
    
    repo_id = config['repo_id']
    
    # Check if model can fit in memory
    is_ok, resource_msg = check_resources(repo_id)
    if not is_ok:
        results.append(create_result_record(
            runtime="llamafile",
            model=repo_id,
            status="SKIPPED",
            error="Insufficient resources: model estimated to exceed available memory",
        ))
        return results
    
    # Download llamafile
    llamafile_path, error = download_llamafile(config)
    if not llamafile_path:
        results.append(create_result_record(
            runtime="llamafile",
            model=repo_id,
            status="FAILED",
            error=error,
        ))
        return results
    
    # Test each prompt
    for prompt_info in PROMPTS:
        prompt_name = prompt_info["name"]
        prompt_text = prompt_info["text"]
        
        try:
            print(f"Running inference on {repo_id} with prompt '{prompt_name}'...")
            log_phase("INFER_START", model=repo_id, prompt=prompt_name)
            
            # Capture memory before inference
            mem_before = get_memory_info()
            mem_before_mb = mem_before['used_gb'] * 1024
            
            start_time = time.time()
            # Run llamafile with prompt (use 'sh' to avoid APE binfmt_misc issues in containers)
            stdout, stderr, returncode, elapsed_time = run_subprocess_with_streaming(
                cmd=["sh", str(llamafile_path), "-p", prompt_text, "--temp", str(TEMPERATURE), "-n", str(MAX_TOKENS), "--repeat-penalty", str(REPETITION_PENALTY), "--no-display-prompt"],
                description=f"llamafile inference for {repo_id}",
                timeout_s=INFERENCE_TIMEOUT_S,
            )
            duration = time.time() - start_time
            
            # Capture memory after inference
            mem_after = get_memory_info()
            mem_after_mb = mem_after['used_gb'] * 1024
            mem_delta_mb = mem_after_mb - mem_before_mb
            
            if returncode == 0:
                log_phase("INFER_DONE", model=repo_id, prompt=prompt_name, duration_s=duration)
                
                # Calculate throughput metrics (using simple token counting)
                prompt_tokens = count_tokens_simple(prompt_text)
                tokens_generated = count_tokens_simple(stdout)
                tokens_per_sec = tokens_generated / duration if duration > 0 else 0.0
                
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=stdout,
                    duration_s=duration,
                    mem_before_mb=mem_before_mb,
                    mem_after_mb=mem_after_mb,
                    mem_delta_mb=mem_delta_mb,
                    prompt_tokens=prompt_tokens,
                    tokens_generated=tokens_generated,
                    tokens_per_sec=tokens_per_sec,
                ))
            else:
                error_msg = f"Execution failed: {stderr[:200]}"
                log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
            results.append(create_result_record(
                runtime="llamafile",
                model=repo_id,
                status="FAILED",
                prompt_name=prompt_name,
                prompt_text=prompt_text,
                error=error_msg,
            ))
        finally:
            clear_memory()
    
    return results


if RUN_LLAMAFILE:
    print("\n" + "="*80)
    print("LLAMAFILE RUNTIME TESTS")
    print("="*80 + "\n")
    
    all_models = LLAMAFILE_MODELS + LLAMAFILE_FALLBACK_MODELS
    
    for config in all_models:
        repo_id = config['repo_id']
        print(f"\nTesting: {repo_id}")
        results = test_llamafile_model(config)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ llamafile tests skipped")


## GPT-OSS-20B Runtime Test (Transformers)

Test OpenAI's GPT-OSS-20B via Hugging Face Transformers.

- **Model:** `openai/gpt-oss-20b` — 20B-parameter MoE (32 experts, 4 active per token)
- **License:** Apache 2.0 (open, no gating)
- **Quantization:** Ships in mxfp4 (~10-12 GB); requires `transformers >= 4.55`

**Success criteria:**
- Model downloads and loads within memory budget
- Inference completes within timeout for each prompt
- Token throughput is recorded for comparison with Gemma models

In [ ]:
# ============================================================================
# GPT-OSS-20B  —  OpenAI's open MoE model via HF Transformers
# https://huggingface.co/openai/gpt-oss-20b
# ============================================================================

RUN_GPT_OSS = True  # flip to False to skip

GPT_OSS_MODEL_ID = "openai/gpt-oss-20b"
GPT_OSS_ESTIMATED_GB = 12.0  # mxfp4 checkpoint ≈ 10-12 GB in RAM


def test_gpt_oss_model(model_id: str) -> List[Dict[str, Any]]:
    """Test OpenAI GPT-OSS-20B with all prompts (Transformers runtime)."""
    results = []
    model = None
    tokenizer = None

    # ── Resource check (custom estimate for 20B MoE) ──
    vm = psutil.virtual_memory()
    available_gb = vm.available / (1024**3)
    threshold_gb = available_gb * 0.8

    print(f"\n{'='*60}")
    print(f"Resource Check: {model_id}")
    print(f"{'='*60}")
    print(f"Estimated model size: {GPT_OSS_ESTIMATED_GB:.1f} GB (mxfp4)")
    print(f"Available RAM: {available_gb:.1f} GB / {vm.total / (1024**3):.1f} GB total")
    print(f"Threshold (80% of available): {threshold_gb:.1f} GB")

    if GPT_OSS_ESTIMATED_GB > threshold_gb:
        msg = (
            f"🛑 BLOCKED: Model size ({GPT_OSS_ESTIMATED_GB:.1f} GB) "
            f"exceeds 80% of available RAM ({threshold_gb:.1f} GB)"
        )
        print(msg)
        print(f"{'='*60}\n")
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error=msg,
        ))
        return results

    print(f"✅ Resources OK: {GPT_OSS_ESTIMATED_GB:.1f} GB needed, {available_gb:.1f} GB available")
    print(f"{'='*60}\n")

    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        # Determine device
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

        log_phase("LOAD_START", model=model_id, device=device, memory=get_memory_info())

        # ── Load model ──
        print(f"Loading {model_id} …")
        start_time = time.time()
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
            # mxfp4 checkpoint auto-dequantizes to bf16 on CPU;
            # we must load as bf16 explicitly to avoid float32/bfloat16 mismatch
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                torch_dtype=torch.bfloat16,
                device_map="auto" if device == "cuda" else None,
                low_cpu_mem_usage=True,
            )
            if device != "cuda":
                model = model.to(device)
            model.eval()
            load_duration = time.time() - start_time
            log_phase("LOAD_DONE", model=model_id, duration_s=load_duration, memory=get_memory_info())
            print(f"✅ Model loaded in {load_duration:.1f}s")
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("LOAD_FAILED", model=model_id, error=error_msg)
            results.append(create_result_record(
                runtime="transformers",
                model=model_id,
                status="FAILED",
                error=error_msg,
                device=device,
            ))
            return results

        # ── Inference for each prompt ──
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]

            try:
                print(f"Running inference on {model_id} with prompt '{prompt_name}'…")
                log_phase("INFER_START", model=model_id, prompt=prompt_name)

                mem_before = get_memory_info()
                mem_before_mb = mem_before["used_gb"] * 1024

                start_time = time.time()
                inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

                with torch.no_grad():
                    with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=(device == "cpu")):
                        outputs = model.generate(
                            **inputs,
                            max_new_tokens=MAX_TOKENS,
                            temperature=TEMPERATURE,
                            do_sample=True,
                            repetition_penalty=REPETITION_PENALTY,
                        )

                output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                duration = time.time() - start_time

                mem_after = get_memory_info()
                mem_after_mb = mem_after["used_gb"] * 1024
                mem_delta_mb = mem_after_mb - mem_before_mb
                log_phase("INFER_DONE", model=model_id, prompt=prompt_name, duration_s=duration)

                # Throughput metrics
                prompt_tokens = len(inputs["input_ids"][0])
                output_tokens = len(outputs[0])
                tokens_generated = output_tokens - prompt_tokens
                tokens_per_sec = tokens_generated / duration if duration > 0 else 0.0
                tokens_per_sec = tokens_generated / duration if duration > 0 else 0.0
                print(f"  ✅ {tokens_generated} tokens in {duration:.1f}s ({tokens_per_sec:.1f} tok/s)")

                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=output_text,
                    duration_s=duration,
                    device=device,
                    mem_before_mb=mem_before_mb,
                    mem_after_mb=mem_after_mb,
                    mem_delta_mb=mem_delta_mb,
                    prompt_tokens=prompt_tokens,
                    tokens_generated=tokens_generated,
                    tokens_per_sec=tokens_per_sec,
                ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                    device=device,
                ))

    except ImportError as e:
        error_msg = f"Import error: {str(e)}"
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error=error_msg,
        ))
    finally:
        del model
        del tokenizer
        clear_memory()
        clear_memory()
    return results


# ── Run GPT-OSS-20B test ──
if RUN_GPT_OSS:
    print("\n" + "=" * 80)
    print("GPT-OSS-20B RUNTIME TEST (Transformers)")
    print("=" * 80 + "\n")

    print(f"Testing: {GPT_OSS_MODEL_ID}")
    results = test_gpt_oss_model(GPT_OSS_MODEL_ID)
    for result in results:
        save_result(result)
        print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
    print()

else:
    print("⚠ GPT-OSS-20B test skipped (disabled in config)")

## Results Summary

Load and display test results.

In [ ]:
# Load results
if RESULTS_JSONL.exists():
    results_data = []
    with open(RESULTS_JSONL, 'r') as f:
        for line in f:
            results_data.append(json.loads(line))
    
    print(f"Loaded {len(results_data)} result records")
    
    # Create DataFrame
    df = pd.DataFrame(results_data)
    
    # Save CSV
    df.to_csv(RESULTS_CSV, index=False)
    print(f"✓ Saved results to {RESULTS_CSV}")
    
    # Display summary
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80 + "\n")
    
    print("Status counts:")
    print(df['status'].value_counts())
    print()
    
    print("Results by runtime:")
    print(df.groupby(['runtime', 'status']).size())
    print()
    
    if len(df[df['status'] == 'SUCCESS']) > 0:
        print("Successful runs - average duration:")
        success_df = df[df['status'] == 'SUCCESS']
        print(success_df.groupby(['runtime', 'model'])['duration_s'].mean())
else:
    print("⚠ No results file found")


## Sovereign Deployment Assessment Report

Generate a comprehensive deployment assessment using OpenAI API.

**Report covers:**
- Performance analysis with quantitative metrics from benchmark data
- Quantization strategy recommendations (fp32, bf16, GGUF Q4/Q6/Q8)
- Deployment venue analysis with cost estimates (local, cloud, edge, K8s)
- Runtime comparison matrix (Transformers, Ollama, llamafile, vLLM)
- Sovereign deployment architecture for agentic systems
- Actionable next steps for production deployment

In [ ]:
if RUN_OPENAI_REPORT and RESULTS_JSONL.exists():
    print("\n" + "="*80)
    print("GENERATING OPENAI REPORT")
    print("="*80 + "\n")
    
    try:
        from openai import OpenAI
        
        # Load all results
        results_data = []
        with open(RESULTS_JSONL, 'r') as f:
            for line in f:
                results_data.append(json.loads(line))
        
        # Estimate token count
        prompt_est_tokens = len(json.dumps(results_data)) // 4
        print(f"\nPreparing OpenAI API call...")
        print(f"  Sending {len(results_data)} results (~{prompt_est_tokens} tokens)")
        print(f"  Model: {OPENAI_MODEL}")
        print(f"  This may take 30-60s...")
        
        # Prepare prompt for OpenAI
        prompt = f"""You are producing a professional technical report for an engineering team evaluating open-weight LLMs for sovereign, self-hosted agentic systems.

## Context

**Test Environment:**
{json.dumps(env_fingerprint, indent=2)}

**Benchmark Results ({len(results_data)} test runs across multiple runtimes and models):**
{json.dumps(results_data, indent=2)}

## Report Requirements

Produce a comprehensive report (1500–2000 words) titled **"Open-Weight LLM Deployment Assessment for Sovereign Agentic Systems"** with the following sections:

### 1. Executive Summary
- Which models successfully ran, which failed, and why.
- Best-performing runtime + model combination for this hardware class.
- One-paragraph verdict on feasibility of sovereign local inference.

### 2. Performance Analysis
- **Table 1:** For every tested model × runtime combination, show: status, inference duration (s), tokens generated, tokens/sec, peak memory usage (MB). Mark the best performer in each category.
- Identify which models are practical for interactive use (<2s first-token latency) vs. batch/offline use.

### 3. Quantization Strategy Recommendations
- Compare the tested precision formats (fp32, bf16, mxfp4, GGUF Q4/Q6/Q8) based on observed results.
- For each model family tested, recommend the optimal quantization for CPU-only machines with 16 GB, 32 GB, and 64 GB RAM.
- Explain when GGUF (llama.cpp/Ollama) quantization is preferable to Transformers bf16/fp32.
- Note any models where quantization degraded output quality (if observable from the test outputs).

### 4. Deployment Venue Analysis
Evaluate these deployment options for running the tested models in production for small agentic systems. For each venue provide: estimated monthly cost, pros, cons, and a suitability rating (★ to ★★★★★).

| Venue | Description |
|-------|-------------|
| **Local workstation** | Developer laptop/desktop (16–64 GB RAM, no GPU) |
| **GitHub Codespaces / Gitpod** | Cloud dev environments (up to 64 GB RAM) |
| **Bare-metal server** | Dedicated Hetzner/OVH box (64–128 GB RAM, optional GPU) |
| **Cloud GPU instance** | AWS g5/p4, GCP a2/g2, Azure NC-series (on-demand or spot) |
| **Edge / Raspberry Pi 5** | ARM SBC with 8 GB RAM |
| **Self-hosted Kubernetes** | K3s/K8s cluster with CPU-only nodes |

Include a **cost comparison table** with approximate monthly costs for running a 7B model 24/7 on each venue.

### 5. Runtime Comparison Matrix
- **Table 2:** Compare Transformers, Ollama, llamafile, and vLLM (theoretical) across: ease of setup, CPU performance, GPU performance, quantization support, API compatibility (OpenAI-compatible), container-friendliness, and memory efficiency.
- Recommend which runtime to standardize on for agentic tool-use workloads.

### 6. Sovereign Deployment Architecture
- Propose a minimal production architecture for running open-weight models on-premises for an agentic system (tool-calling, RAG, multi-step reasoning).
- Cover: model serving (Ollama vs. vLLM vs. llamafile), API gateway, monitoring, model update strategy, and security considerations.
- Recommend the smallest viable model size for reliable agentic tool-use (function calling, JSON output, multi-turn).

### 7. Actionable Recommendations
Provide a prioritized list of 5–7 concrete next steps, such as:
- Which model + runtime + quantization to deploy first
- Hardware upgrades that would unlock better performance (e.g., adding a GPU)
- Which models to skip for this hardware class
- Monitoring/observability to add for production inference

### Formatting
- Use Markdown with clear headers, tables, and bullet points.
- Be quantitative — cite specific numbers from the benchmark data.
- Be opinionated — make clear recommendations, not just options.
- Write for a technical audience building production systems."""
        
        print("Calling OpenAI API...")
        
        try:
            import httpx
            timeout_client = OpenAI(
                api_key=OPENAI_API_KEY,
                timeout=httpx.Timeout(180.0, connect=30.0)  # 3 min for longer report
            )
            
            response = timeout_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": (
                        "Act as a senior ML infrastructure engineer and solutions architect "
                        "specializing in on-premises and sovereign AI deployments. Write "
                        "precise, data-driven technical reports with concrete cost estimates "
                        "and actionable recommendations. Favor quantized open-weight models "
                        "served via efficient runtimes (Ollama, vLLM, llama.cpp) for production "
                        "workloads. The audience is engineering leadership evaluating self-hosted "
                        "LLM inference for agentic applications."
                    )},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=4096,
            )
            
            # Print usage stats
            if hasattr(response, 'usage') and response.usage:
                print(f"\n✓ OpenAI API call successful!")
                print(f"  Prompt tokens: {response.usage.prompt_tokens}")
                print(f"  Completion tokens: {response.usage.completion_tokens}")
                print(f"  Total tokens: {response.usage.total_tokens}")
        
        except Exception as e:
            error_type = type(e).__name__
            print(f"\n❌ OpenAI API error ({error_type}): {e}")
            if "timeout" in str(e).lower() or "TimeoutError" in error_type:
                print("The API call timed out after 120s. Try again later or check your connection.")
            elif "rate" in str(e).lower() or "RateLimitError" in error_type:
                print("Rate limit exceeded. Wait a few minutes before trying again.")
            elif "connection" in str(e).lower() or "ConnectionError" in error_type:
                print("Could not connect to OpenAI API. Check your internet connection.")
            raise
        
        report_text = response.choices[0].message.content
        
        # Save markdown report
        with open(REPORT_MD, 'w') as f:
            f.write(report_text)
        print(f"✓ Saved report to {REPORT_MD}")
        
        # Save JSON report
        report_json = {
            "generated_at": datetime.now().isoformat(),
            "model_used": OPENAI_MODEL,
            "environment": env_fingerprint,
            "report_text": report_text,
            "results_summary": {
                "total_tests": len(results_data),
                "successful": len([r for r in results_data if r['status'] == 'SUCCESS']),
                "failed": len([r for r in results_data if r['status'] == 'FAILED']),
                "skipped": len([r for r in results_data if r['status'] == 'SKIPPED']),
            }
        }
        
        with open(REPORT_JSON, 'w') as f:
            json.dump(report_json, f, indent=2)
        print(f"✓ Saved report metadata to {REPORT_JSON}")
        
        # Display report as rendered Markdown in notebook output
        from IPython.display import display, Markdown
        print("\n" + "="*80)
        print("REPORT")
        print("="*80 + "\n")
        display(Markdown(report_text))
        
    except Exception as e:
        print(f"⚠ Report generation failed: {type(e).__name__}: {str(e)}")
elif not RUN_OPENAI_REPORT:
    print("⚠ OpenAI report generation skipped (disabled in config)")
else:
    print("⚠ No results file found - skipping report generation")


## Cleanup: Remove Secrets from Notebook

**Run this cell before committing to git!**

This strips all outputs, metadata, and environment variables that may contain API keys or tokens from the notebook and kernel memory.

In [ ]:
# ============================================================================
# 🧹 CLEANUP — Run this cell before `git add` / `git commit`!
# ============================================================================
# This cell:
#   1. Wipes API keys from kernel memory and os.environ
#   2. Strips all cell outputs from this notebook file on disk
#   3. Removes execution counts and widget metadata
#
# After running, save the notebook (Ctrl+S) then commit safely.
# ============================================================================

import json
from pathlib import Path

NOTEBOOK_PATH = Path("gemma_runtime_smoketest.ipynb")

# ── Step 1: Clear secrets from kernel memory ──
_secrets_cleared = []
for _var in ['HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'OPENAI_API_KEY']:
    if _var in os.environ:
        del os.environ[_var]
        _secrets_cleared.append(_var)
    # Also clear Python-level variables
    if _var in dir():
        exec(f"{_var} = None")

# Clear module-level refs
try:
    HF_TOKEN = None
    OPENAI_API_KEY = None
except:
    pass

if _secrets_cleared:
    print(f"✅ Cleared from os.environ: {', '.join(_secrets_cleared)}")
else:
    print("ℹ  No secrets found in os.environ")

# ── Step 2: Strip outputs from the notebook file on disk ──
if NOTEBOOK_PATH.exists():
    with open(NOTEBOOK_PATH, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    _cells_cleaned = 0
    for cell in nb.get('cells', []):
        # Clear all outputs
        if cell.get('outputs'):
            cell['outputs'] = []
            _cells_cleaned += 1
        # Reset execution count
        if 'execution_count' in cell:
            cell['execution_count'] = None
        # Remove widget metadata
        if 'metadata' in cell:
            cell['metadata'].pop('widgets', None)
            cell['metadata'].pop('execution', None)

    # Remove notebook-level widget state
    nb.get('metadata', {}).pop('widgets', None)

    with open(NOTEBOOK_PATH, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
        f.write('\n')

    print(f"✅ Stripped outputs from {_cells_cleaned} cells in {NOTEBOOK_PATH}")
    print(f"   Notebook is now safe to commit.")
else:
    print(f"⚠️  Notebook not found at {NOTEBOOK_PATH}")

print("\n📋 Pre-commit checklist:")
print("   1. ✅ Secrets removed from memory")
print("   2. ✅ Cell outputs stripped from notebook")
print("   3. ➡️  Save the notebook (Ctrl+S / Cmd+S)")
print("   4. ➡️  Verify with: git diff --cached")
print("   5. ➡️  Commit: git add . && git commit")

## Test Complete

All tests have been executed. Check the `artifacts/` directory for:
- `results.jsonl` - Raw test results
- `results.csv` - Flattened summary
- `report.md` - Human-readable analysis
- `report.json` - Machine-readable report metadata